# Guía para la clase de desarrollo de software
---

## Fundamentos del diseño

### Fundamentos del Ciclo de Vida

El Software Development Life Cycle (SDLC) es el marco metodológico esencial para optimizar los flujos de trabajo. Su implementación correcta permite transformar una necesidad de negocio en un entregable técnico de alta calidad mediante etapas estrictamente definidas: análisis, diseño, desarrollo, pruebas, implantación y la generación de entregables.

Se compone de:

* Requisitos: Identificación de las necesidades del usuario y objetivos de negocio.
* Diseño: Planificación de la arquitectura técnica y la interfaz de usuario.
* Programación: Construcción del código fuente de la funcionalidad.
* Pruebas: Validación rigurosa de la carga y respuesta de la interfaz.
* Despliegue: Lanzamiento controlado de la función al entorno de producción.
* Mantenimiento: Monitoreo continuo y corrección de errores post-lanzamiento.

Resumen: Orden de operaciones

### La Cultura DevOps

DevOps representa la convergencia entre el desarrollo (Dev) y las operaciones de TI (Ops). Es una cultura que se rige bajo los siguientes pilares:

* Automatización total: Eliminación de procesos manuales propensos al error humano.
* Detección de bugs en segundos: Retroalimentación inmediata del estado del código.
* CI/CD (Integración y Entrega Continua): Flujos automáticos que validan, prueban y publican versiones.
* Infraestructura como Código (IaC): Configuración de despliegues mediante scripts, garantizando la reproducibilidad.

Resumen: Arquitectura técnica robusta para soportar la carga y complejidad de grandes proyectos


---

## Arquitectura de Software

### Arquitectura de Software
* **Frontend**: La interfaz que gestiona la interacción del usuario
* **Backend**: El motor que procesa la lógica y el almacenamiento

Ambos entornos pueden desplegarse dos maneras:
* **Monolítica**: Código único e indivisible. Solo puede manejarse (para escalabilidad y despliegue) como un bloque único. Un fallo hace que el sistema entero colapse. Se recomienda para proyectos pequeños o prototipos iniciales.
* **Microservicios**: Descomposición en servicios independientes. Puede manejarse (para escalabilidad y despliegue) como bloques individuales. Un fallo se contienen en el servicio. Se recomienda para aplicaciones complejas

### Base de datos
* **Bases de Datos Relacionales (SQL)**: Basadas en tablas con llaves primarias y foráneas. Garantizan la integridad referencial. Ejemplo: MySQL.
* Bases de Datos NoSQL: Ofrecen esquemas flexibles para datos desestructurados mediante documentos JSON/BSON. Ideales para escalabilidad horizontal. Ejemplo: MongoDB.

### Cloud Computing
La **computación en la nube** permite el acceso bajo demanda a recursos de TI. Su mayor ventaja es la elasticidad y la alta disponibilidad. Las instancias Amazon EC2 son el estándar para servidores virtuales que ajustan su potencia según el tráfico real.

---


## Ingeniería de Datos: Pipelines y Rendimiento
La estrategia de procesamiento es clave para poder tener una ingeniería de datos eficiente y verdaderamente útil.

### Diferencia entre ETL y ELT
La diferencia radica en el orden de los pasos y el lugar físico donde se transforman los datos:

#### ELT (Extract, Load, Transform)
*   **Concepto:** Los datos **se almacenan primero** en bruto en la plataforma de destino y **se transforman directamente dentro de ella** aprovechando su propio motor.
*   **Flujo detallado:**
    1.  **Extract (Extracción):** Se leen los datos directamente desde el origen.
    2.  **Load (Carga):** Se insertan los datos crudos o semiestructurados directamente en la base de datos o repositorio de destino.
    3.  **Transform (Transformación):** Se ejecutan las consultas de limpieza, conversión de tipos y lógica de negocio directamente dentro de la plataforma final.
Ejemplo: El ejercicio donde se carga primero el CSV crudo directamente en un archivo de **DuckDB** y se realizó posteriormente la validación y transformación ejecutando sentencias **SQL dentro de DuckDB**.

#### ETL (Extract, Transform, Load)
*   **Concepto:** Los datos **se transforman antes de cargarlos** en el destino final.
*   **Flujo detallado:**
    1.  **Extract (Extracción):** Se obtiene la información de múltiples fuentes heterogéneas (como SQLite, JSON, APIs o CSV) garantizando la trazabilidad (registro de origen y fecha).
    2.  **Transform (Transformación):** Los datos se limpian, filtran, homologan (por ejemplo, corregir alias de marcas) y se combinan (*merge*), además de calcular variables derivadas en memoria temporal.
    3.  **Load (Carga):** Los datos, ya limpios y procesados, se persisten en el destino final.
Ejemplo: El ejercicio de vehículos de carga y transformación de la información en memoria usando Pandas antes de guardar el resultado limpio en la base de datos final `vehicles_curated`.

In [14]:
# Construcción básica de un ETL
from pathlib import Path
import pandas as pd

# Importar dataset
DATA_PATH = Path.cwd() / "../Machine Learning/Tareas/02-Supervised-Machine-Learning" /"seattle-weather-dataset.csv"
data = pd.read_csv(DATA_PATH)
df = pd.DataFrame(data)

# 2. TRANSFORM
df = df.dropna()                    # eliminar datos faltantes
df["weather"] = df["weather"].str.upper() # cambiar tipo de dato

# 3. LOAD
# df.to_csv("dataset_prueba.csv", index=False)

df.head(3)

,date,precipitation,temp_max,temp_min,wind,weather
0,1/1/2012,0.0,12.8,5.0,4.7,DRIZZLE
1,1/2/2012,10.9,10.6,2.8,4.5,RAIN
2,1/3/2012,0.8,11.7,7.2,2.3,RAIN


In [ ]:
# Ejemplo más completo
s
import pandas as pd

# =========================
# 1. EXTRACT
# =========================

ventas = pd.read_csv("ventas.csv")
clientes = pd.read_csv("clientes.csv")
productos = pd.read_csv("productos.csv")


# =========================
# 2. TRANSFORM
# =========================

# Eliminar registros incompletos
ventas = ventas.dropna(subset=["cliente_id", "producto_id", "cantidad"])

# Convertir tipos de datos
ventas["cantidad"] = ventas["cantidad"].astype(int)
ventas["precio"] = ventas["precio"].astype(float)

# Normalizar nombres de columnas
clientes.columns = clientes.columns.str.lower().str.strip()

# Unir información de clientes
ventas = ventas.merge(
    clientes[["cliente_id", "estado"]],
    on="cliente_id",
    how="left"
)

# Unir información de productos
ventas = ventas.merge(
    productos[["producto_id", "categoria"]],
    on="producto_id",
    how="left"
)

# Crear una variable derivada
ventas["total"] = ventas["cantidad"] * ventas["precio"]

# Filtrar ventas inválidas
ventas = ventas[ventas["total"] > 0]

# Homologar valores
ventas["estado"] = ventas["estado"].str.upper().str.strip()


# =========================
# 3. LOAD
# =========================

ventas.to_csv(
    "ventas_procesadas.csv",
    index=False
)

print("ETL completado")


## Consideraciones importantes
### Estrategias de Carga e Idempotencia

La idempotencia es una propiedad fundamental: el resultado debe ser el mismo sin importar cuántas veces se ejecute el script.

1. Full Load: Reconstrucción total de la tabla en cada ejecución (Ej: reemplazo total).
2. Incremental Load: Procesa solo datos nuevos o modificados mediante un Watermark (updated_at). Se logra mediante la técnica UPSERT basada en una business key (clave de negocio) estable.

### Data Quality y Validación (Checkpoints)

La validación técnica es el "Quality Gate" antes de la persistencia. Debemos implementar umbrales específicos:

* Validación de Cardinalidad: Es crítico evitar que un JOIN (1:1) se convierta en (1:N) debido a duplicados en la business key. Ejemplo numérico: Si tenemos 10,000 órdenes y el catálogo de clientes tiene duplicados, el JOIN multiplicará las filas, corrompiendo los reportes financieros.
* Reglas de Negocio para Nulos: En el caso de Telco Churn, valores nulos en TotalCharges para clientes con tenure=0 no son errores, sino clientes nuevos. La transformación debe normalizarlos a 0.0.
* Estrategia de Cuarentena: Los registros que no superan umbrales (ej. completeness_min: 0.9) deben moverse a una zona aislada para auditoría, nunca eliminarse silenciosamente.

### Rendimiento: El Crossover Point (Pandas vs. Spark)

Existe un punto de inflexión técnica donde la herramienta cambia:

* **Pandas**: Superior en volúmenes pequeños (< 10 millones de filas) al ejecutar directamente en memoria Python, evitando la traducción de código.
* **Apache Spark**: Indispensable a partir de los 10 millones de filas (~320 MB - 650 MB). A pesar del Overhead de la JVM y la inicialización del SparkContext, Spark utiliza el Catalyst Optimizer para planificar consultas eficientes.

Formato Parquet: Superior al CSV por ser columnar, incluir metadatos y poseer esquemas embebidos, optimizando drásticamente el tiempo de consulta.

Una vez procesados los datos, la robustez del sistema depende de la calidad del código fuente y su correcta verificación.

### Calidad de Software: Pruebas y Documentación

Garantizar la calidad requiere pruebas sistemáticas utilizando herramientas como Jest.

Niveles de Pruebas (Caso Dittravel)

1. Pruebas Unitarias: Validación aislada de una función. Ejemplo: Verificar que la recuperación de contraseña valide el formato del correo.
2. Pruebas de Integración: Validación del flujo completo entre componentes. Ejemplo: El proceso de login, desde el hasheo de credenciales hasta la validación en la base de datos del backend.

El Contrato Técnico: README

El archivo README no es opcional; es el estándar que define las reglas de contribución, instrucciones de instalación y el alcance técnico del proyecto.

La robustez del software y la integridad de los datos son los cimientos necesarios para el despliegue de modelos de inteligencia artificial.

## Machine Learning y Métricas de Evaluación

El flujo de un sistema de ML es lineal pero riguroso: Datos -> Preprocesamiento -> Entrenamiento -> Modelo -> Evaluación.

### Algoritmos y Ajustes

* Regresión Lineal: Para predicción de variables numéricas continuas.
* Regresión Logística: Específicamente diseñada para clasificación binaria mediante una función sigmoide.
* KNN: Clasificación por cercanía en el espacio de datos.

### Ajuste de Variables (Escalamiento):

* Normalización: Rango 0 a 1.
* Estandarización: Media cero (\mu=0) y desviación estándar uno (\sigma=1).
* Regla de Oro: "La transformación aplicada en el entrenamiento debe ser idéntica en la inferencia". Ignorar esto invalida cualquier predicción del modelo.

### Métricas de Regresión

* MAE (Error Absoluto Medio): Promedio de las diferencias reales.
* RMSE (Raíz del Error Cuadrático Medio): Magnifica y penaliza errores grandes; indica la significancia del fallo.
* R² (Coeficiente de Determinación): Indica qué tanta variabilidad explica el modelo (el "Accuracy" de la regresión).